# 01 - Data check

What is in the data before any model is trained: image types, nuclei per image, nucleus size, touching nuclei.
Output: `out/image_stats.csv`, `out/split.csv` and two figures. Runs on CPU; Kaggle needs Internet on.

In [ ]:
import glob, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io
from skimage.segmentation import find_boundaries
from sklearn.model_selection import train_test_split

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('.')
# raw data goes to /tmp so it is not saved as notebook output (only out/ is kept)
DATA = Path('/tmp/nuclei_data')
OUT = WORK / 'out'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)

URL = 'https://data.broadinstitute.org/bbbc/BBBC038/stage1_train.zip'
zip_path = DATA / 'stage1_train.zip'
train_dir = DATA / 'stage1_train'
if not zip_path.exists():
    urllib.request.urlretrieve(URL, zip_path)
if not train_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(train_dir)

img_paths = sorted(glob.glob(str(train_dir / '**' / 'images' / '*.png'), recursive=True))
print(len(img_paths), 'images')

Each image folder has one PNG per nucleus in `masks/`. `load` puts them into one label image: 0 is background, 1..N are the nuclei.
`overlap_px` counts pixels claimed by two masks (should be 0).

In [ ]:
def load(img_path):
    img = io.imread(img_path)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    img = img[..., :3]
    mask_paths = sorted(glob.glob(str(Path(img_path).parent.parent / 'masks' / '*.png')))
    label = np.zeros(img.shape[:2], np.int32)
    count = np.zeros(img.shape[:2], np.int32)
    for k, mp in enumerate(mask_paths, start=1):
        m = io.imread(mp) > 0
        label[m] = k
        count += m
    return img, label, int((count > 1).sum())


def image_type(img):
    # colour stain -> histology; grey with dark background -> fluorescence; grey with bright background -> brightfield
    x = img.astype(np.float32)
    colour = np.abs(x[..., 0] - x[..., 1]).mean() + np.abs(x[..., 1] - x[..., 2]).mean()
    if colour > 5:
        return 'histology'
    return 'fluorescence' if x.mean() < 100 else 'brightfield'


def touching_fraction(label):
    # share of nuclei that touch at least one other nucleus (4-neighbourhood)
    pairs = set()
    for a, b in [(label[:, :-1], label[:, 1:]), (label[:-1, :], label[1:, :])]:
        m = (a > 0) & (b > 0) & (a != b)
        pairs.update(zip(a[m].tolist(), b[m].tolist()))
    touching = {x for p in pairs for x in p}
    n = len(np.unique(label)) - 1
    return len(touching) / n if n else 0.0


def three_class(label):
    # target for the model: 0 background, 1 nucleus interior, 2 nucleus boundary
    t = np.zeros(label.shape, np.uint8)
    t[label > 0] = 1
    t[find_boundaries(label, mode='inner') & (label > 0)] = 2
    return t

In [ ]:
rows = []
for p in img_paths:
    img, label, overlap = load(p)
    areas = np.bincount(label.ravel())[1:]
    areas = areas[areas > 0]
    rows.append(dict(
        id=Path(p).stem, path=p, type=image_type(img),
        height=img.shape[0], width=img.shape[1],
        n_nuclei=len(areas),
        median_area=float(np.median(areas)) if len(areas) else 0.0,
        min_area=int(areas.min()) if len(areas) else 0,
        touching_frac=touching_fraction(label),
        overlap_px=overlap,
    ))
df = pd.DataFrame(rows)
print(df.type.value_counts())
print('images with overlapping masks:', (df.overlap_px > 0).sum())
df.groupby('type')[['n_nuclei', 'median_area', 'touching_frac', 'height', 'width']].median()

Split 80/20 by image, stratified by type, fixed seed. Later notebooks use `split.csv`, so every model sees the same images.

In [ ]:
train_ids, val_ids = train_test_split(df.id, test_size=0.2, stratify=df.type, random_state=0)
df['split'] = np.where(df.id.isin(set(val_ids)), 'val', 'train')
df.drop(columns='path').to_csv(OUT / 'image_stats.csv', index=False)
df[['id', 'type', 'split']].to_csv(OUT / 'split.csv', index=False)
df.groupby(['type', 'split']).size().unstack()

Check the type rule by eye: four random images per type with the expert boundaries in yellow. If an image is in the wrong row, the rule needs fixing before training.

In [ ]:
types = ['fluorescence', 'brightfield', 'histology']
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for r, t in enumerate(types):
    sub = df[df.type == t]
    picks = sub.sample(min(4, len(sub)), random_state=0)
    for c in range(4):
        ax = axes[r, c]
        ax.axis('off')
        if c >= len(picks):
            continue
        img, label, _ = load(picks.iloc[c].path)
        show = img.copy()
        show[find_boundaries(label, mode='inner')] = [255, 255, 0]
        ax.imshow(show)
        ax.set_title(f"{t} | {picks.iloc[c].n_nuclei} nuclei", fontsize=10)
plt.tight_layout()
plt.savefig(OUT / 'examples_by_type.png', dpi=110)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for t in types:
    vals = df[df.type == t].median_area
    if len(vals):
        ax.hist(vals, bins=np.logspace(1, 4, 40), alpha=0.6, label=f"{t} ({len(vals)} images)")
ax.set_xscale('log')
ax.set_xlabel('median nucleus area per image (pixels)')
ax.set_ylabel('images')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'nucleus_area_by_type.png', dpi=110)
plt.show()

In [ ]:
# quick check of the training target on one image: 0 background, 1 interior, 2 boundary
img, label, _ = load(df.iloc[0].path)
t = three_class(label)
print('pixel share per class:', np.bincount(t.ravel(), minlength=3) / t.size)

All results in one small zip, so the Kaggle output can be downloaded as a single file.

In [ ]:
import shutil
shutil.make_archive(str(WORK / 'results_01'), 'zip', OUT)
print(sorted(p.name for p in OUT.iterdir()))